# Component 3 - inference normalisation fix (Phase 2 step 2.7 re-run + Phase 3 extraction)

Model input is now normalised over the **lit tissue only** (as the fully-lit training crops were), not over
the whole image incl. the dark area. Same final models - **no retraining**. Old results are kept
(`seg/full_val/`, `extraction/full_val/`); new ones go to `..._litnorm/`.

**Runtime > Change runtime type > L4 GPU** (cells 3-4); the rest also runs on CPU.

In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Packages (Colab only) + unit tests
!pip -q install "cellpose>=4.0,<5" tensorflow tf-keras stardist
!nvidia-smi -L
!python -m pytest tests/test_extraction.py tests/test_normalise.py -q

In [ ]:
# 3. Re-segment the 197 validation images with lit-area normalisation (~15 min on L4)
!python -m segmentation.infer_full --run cellpose_finetune_flat 2>&1 | grep -E "images|done|Error|error"
!python -m segmentation.infer_full --run stardist_finetune_flat 2>&1 | grep -E "images|done|Error|error"

In [ ]:
# 4. Step 2.7 comparison with Konan: AFTER the fix, then BEFORE (first run) for reference
!python -m segmentation.infer_full --compare
print("\n================ before the fix (whole-image normalisation) ================")
!python -m segmentation.infer_full --compare --norm image

In [ ]:
# 5. Phase 3 extraction on the new masks + HEX / CV vs Konan + open-ring sensitivity (CPU work)
!python -m extraction.run_extract
!python -m extraction.run_extract --compare
!python -m extraction.run_extract --open-sensitivity 0.10 0.20 0.30

In [ ]:
# 6. Figures: cells coloured by side count, same 3 images as before
import pandas as pd
from IPython.display import Image, display
man = pd.read_csv("splits/manifest.csv")
val = man[(man["split"] == "val") & (man["status"] == "included")]
ids = [g.sample(1, random_state=0)["image_id"].iloc[0] for _, g in val.groupby("ecd_band")]
ARGS = " ".join('"' + i + '"' for i in ids)          # image IDs contain spaces
!python -m extraction.run_extract --figure {ARGS}
for i in ids:
    display(Image(f"{OUTPUT_ROOT}/extraction/full_val_litnorm/stardist_finetune_flat/figures/{i}.png", width=1000))